In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# WEEK 12 — FUNCTION 4 (PCA LENS v7)
# Key upgrades vs Week 11:
#  1) Weighted PCA on X (weights favor higher y) to find principal directions
#  2) Candidate generation happens mostly in low-d PCA subspace (PC1..PCd)
#     -> reduces randomness but preserves exploration (small orthogonal noise)
#  3) Redundancy reduction: penalize candidates too close to existing points
#     in PCA space (not just raw space)
#  4) Still domain-safe: x in [0,1]^4
#  5) x_next printed to 6 decimals or less
# ============================================================

# ----------------------------
# 0) Helpers
# ----------------------------
def clamp01(a):
    return np.minimum(1.0, np.maximum(0.0, a))

def fmt_x6(x):
    return "[" + ", ".join(f"{float(v):.6f}" for v in x) + "]"

def is_duplicate(x, X_existing, tol=1e-6):
    return np.any(np.linalg.norm(X_existing - x, axis=1) < tol)

def norm01(v):
    v = np.asarray(v, dtype=np.float64)
    lo, hi = np.min(v), np.max(v)
    return (v - lo) / (hi - lo + 1e-12)

def nearest_dist(cands, X_existing):
    diff = cands[:, None, :] - X_existing[None, :, :]
    d2 = np.sum(diff * diff, axis=2)
    return np.sqrt(np.min(d2, axis=1) + 1e-12)

def topk_nearest(x, X, y, k=3):
    d = np.linalg.norm(X - x[None, :], axis=1)
    idx = np.argsort(d)[:k]
    return idx, d[idx], y[idx]

# ----------------------------
# 1) Input data (Function 4) — RAW
# ----------------------------
X_train_raw = np.array([
    [0.89698105, 0.72562797, 0.17540431, 0.70169437],
    [0.8893564 , 0.49958786, 0.53926886, 0.50878344],
    [0.25094624, 0.03369313, 0.14538002, 0.49493242],
    [0.34696206, 0.0062504 , 0.76056361, 0.61302356],
    [0.12487118, 0.12977019, 0.38440048, 0.2870761 ],
    [0.80130271, 0.50023109, 0.70664456, 0.19510284],
    [0.24770826, 0.06044543, 0.04218635, 0.44132425],
    [0.74670224, 0.7570915 , 0.36935306, 0.20656628],
    [0.40066503, 0.07257425, 0.88676825, 0.24384229],
    [0.6260706 , 0.58675126, 0.43880578, 0.77885769],
    [0.95713529, 0.59764438, 0.76611385, 0.77620991],
    [0.73281243, 0.14524998, 0.47681272, 0.13336573],
    [0.65511548, 0.07239183, 0.68715175, 0.08151656],
    [0.21973443, 0.83203134, 0.48286416, 0.08256923],
    [0.48859419, 0.2119651 , 0.93917791, 0.37619173],
    [0.16713049, 0.87655456, 0.21723954, 0.95980098],
    [0.21691119, 0.16608583, 0.24137226, 0.77006248],
    [0.38748784, 0.80453226, 0.75179548, 0.72382744],
    [0.98562189, 0.66693268, 0.15678328, 0.8565348 ],
    [0.03782483, 0.66485335, 0.16198218, 0.25392378],
    [0.68348638, 0.9027701 , 0.33541983, 0.99948256],
    [0.17034731, 0.75695908, 0.27652049, 0.5312315 ],
    [0.85965692, 0.91959232, 0.20613873, 0.09779683],
    [0.28213837, 0.50598691, 0.53053084, 0.09630162],
    [0.32607578, 0.4723669 , 0.453192  , 0.10588734],
    [0.94838936, 0.89451301, 0.85163782, 0.55219629],
    [0.66495539, 0.04656628, 0.11677747, 0.79371778],
    [0.57776561, 0.42877174, 0.42582587, 0.24900741],
    [0.73861301, 0.48210263, 0.70936644, 0.50397001],
    [0.8548108 , 0.49396462, 0.73530997, 0.80809201],
    [1.085621  , 1.019592  , 1.039177  , 1.099482  ],   # out-of-bounds in raw
    [1.00000e-06, 1.00000e-06, 1.24558e-01, 1.00000e-06],
    [0.866175  , 0.601115  , 0.708072  , 0.020585  ],
    [0.145904  , 0.536548  , 0.6014    , 0.01905   ],
    [0.356293  , 0.442523  , 0.13052   , 0.242559  ],
    [0.061431  , 0.381247  , 0.983792  , 0.705575  ],
    [0.497045, 0.450388, 0.380113, 0.297612],
    [0.480706, 0.444032, 0.354963, 0.354729],
    [0.503198, 0.435617, 0.371338, 0.408316],
    [0.506271, 0.414408, 0.366165, 0.398853],
    [0.492000, 0.438000, 0.345000, 0.370000]
], dtype=float)

y_train = np.array([
    -22.10828779, -14.60139663, -11.69993246, -16.05376511, -10.06963343,
    -15.48708254, -12.68168498, -16.02639977, -17.04923465, -12.74176599,
    -27.31639636, -13.52764887, -16.6791152 , -16.50715856, -17.81799934,
    -26.56182083, -12.75832422, -19.44155762, -28.90327367, -13.70274694,
    -29.4270914 , -11.56574199, -26.85778644,  -7.96677535,  -6.70208925,
    -32.62566022, -19.98949793,  -4.02554228, -13.12278233, -23.1394284 ,
    -67.60493430274798, -22.782193418373407, -22.194212794446454, -13.363105653346768,  -5.926020577803715,
    -23.786955955997737, -2.5615259470796796, -0.81371612670717, -1.1642621740684471, -1.2544758182228928,
    -0.9934706551455466
], dtype=float)

assert len(X_train_raw) == len(y_train), "X_train and y_train length mismatch"

# ----------------------------
# 2) Enforce domain bounds [0,1]^4
# ----------------------------
X_train = clamp01(X_train_raw)

# ----------------------------
# 3) Current best (maximisation)
# ----------------------------
current_best_idx = int(np.argmax(y_train))
current_best_x = X_train[current_best_idx]
current_best_y = float(y_train[current_best_idx])

print("Current best index:", current_best_idx)
print("Current best X (clamped to [0,1]):", current_best_x)
print("Current best y:", current_best_y)

# ----------------------------
# 4) Fixed scaling for [0,1]^4 (raw == scaled)
# ----------------------------
X_scaled = X_train.copy()
y_mean = y_train.mean()
y_std = y_train.std() + 1e-12
y_scaled = (y_train - y_mean) / y_std

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_tensor_all = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor_all = torch.tensor(y_scaled, dtype=torch.float32, device=device).unsqueeze(-1)

torch.manual_seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

# ----------------------------
# 5) Surrogate model (keep, but slightly lighter so it's stable)
# ----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim=4, hidden=(48, 48), p_dropout=0.03):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p_dropout),
            nn.Linear(h2, 1),
        )
    def forward(self, x):
        return self.net(x)

def train_model(model, X, y, max_epochs=650, lr=1.2e-3, weight_decay=8e-6, patience=55, min_delta=1e-4):
    criterion = nn.SmoothL1Loss(beta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best = float("inf")
    bad = 0
    model.train()
    for _ in range(max_epochs):
        optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        lv = float(loss.item())
        if lv < best - min_delta:
            best = lv
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    return best

def train_ensemble(X_tensor, y_tensor, rr, n_ens=5, hidden=(48,48), dropout=0.03):
    ensemble = []
    n = len(X_tensor)
    for m in range(n_ens):
        boot_idx = rr.integers(0, n, size=n)
        Xb = X_tensor[boot_idx]
        yb = y_tensor[boot_idx]
        torch.manual_seed(500 + m)
        model = MLP(input_dim=4, hidden=hidden, p_dropout=dropout).to(device)
        train_model(model, Xb, yb)
        ensemble.append(model)
    return ensemble

ensemble = train_ensemble(X_tensor_all, y_tensor_all, rng, n_ens=5)

# ----------------------------
# 6) Prediction utilities (original y units) + EI/PI
# ----------------------------
def ensemble_predict(ensemble, X_scaled_tensor):
    preds = []
    with torch.no_grad():
        for model in ensemble:
            model.eval()
            p_scaled = model(X_scaled_tensor).squeeze(-1)
            p_raw = p_scaled * y_std + y_mean
            preds.append(p_raw)
    preds = torch.stack(preds, dim=0)
    return preds.mean(dim=0), preds.std(dim=0) + 1e-9

normal = torch.distributions.Normal(
    torch.tensor(0.0, device=device),
    torch.tensor(1.0, device=device)
)

def expected_improvement(mean, std, best_y, xi=0.004):
    imp = mean - best_y - xi
    Z = imp / std
    ei = imp * normal.cdf(Z) + std * torch.exp(normal.log_prob(Z))
    return torch.where(std > 0, ei, torch.zeros_like(ei))

def probability_of_improvement(mean, std, best_y, xi=0.0):
    imp = mean - best_y - xi
    Z = imp / std
    return normal.cdf(Z)

# ----------------------------
# 7) WEEK 12: Weighted PCA (variance focus, remove redundancy)
#    - weights emphasize higher y (better points)
#    - we then generate candidates mostly in top-PC subspace
# ----------------------------
def weighted_pca(X, y, temp=1.8):
    """
    Returns:
      mu: weighted mean (4,)
      V: eigenvectors (4,4) columns sorted by descending eigenvalue
      evals: eigenvalues (4,)
      expl: explained variance ratio (4,)
      W: normalized weights (n,)
    """
    y = np.asarray(y, dtype=np.float64)
    y_max = float(np.max(y))
    w = np.exp((y - y_max) / float(temp))
    w = w / (np.sum(w) + 1e-12)

    mu = np.sum(X * w[:, None], axis=0)
    Xc = X - mu[None, :]
    cov = (Xc * w[:, None]).T @ Xc  # weighted covariance (weights sum to 1)
    evals, V = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    V = V[:, order]
    expl = evals / (np.sum(evals) + 1e-12)
    return mu, V, evals, expl, w

pca_mu, pca_V, pca_evals, pca_expl, pca_w = weighted_pca(X_train, y_train, temp=1.8)

# Use top d PCs (dimension reduction)
PCA_D = 2
Vd = pca_V[:, :PCA_D]  # (4, d)

def to_pca(x_raw01):
    xc = x_raw01 - pca_mu
    return xc @ Vd  # (d,)

def from_pca(z):
    x = pca_mu + (z @ Vd.T)
    return clamp01(x)

# Existing points in PCA space (for redundancy control)
Z_existing = np.array([to_pca(x) for x in X_train], dtype=np.float64)

print("\n=== WEEK 12 PCA SUMMARY ===")
print("Explained variance ratio:", [float(v) for v in pca_expl])
print(f"Using PCA_D = {PCA_D} principal components.")
print("Weighted PCA mean:", fmt_x6(pca_mu))
print("PC1 vector:", fmt_x6(pca_V[:, 0]))
print("PC2 vector:", fmt_x6(pca_V[:, 1]))

# ----------------------------
# 8) Week 12 candidate policy (PCA-guided)
#    - mostly exploit within PCA subspace near the best region
#    - keep small orthogonal noise (reduced randomness)
#    - keep small global spill
# ----------------------------
DECODING = {"max_tokens": 7000}

CAND_MIX = {
    "pca_local_best_frac": 0.72,   # exploit around best in PCA coords
    "pca_local_top_frac": 0.18,    # exploit around weighted-top centroid in PCA coords
    "global_frac": 0.10,           # guard rail
    "pca_trust": 0.075,            # PCA step size
    "orth_noise": 0.012,           # small noise in orthogonal complement
}

REFINE = {
    "topk_seed": 180,
    "per_seed": 14,
    "pca_refine": 0.040,
    "orth_refine": 0.008
}

SCORE_W = {"w_mean": 0.72, "w_ei": 0.24, "w_pi": 0.04}

PEN = {
    "lambda_raw_near": 0.08,       # discourage wild extrapolation in raw space
    "lambda_pca_redund": 0.12,     # discourage redundancy in PCA space
    "min_pca_sep": 0.010,          # hard-ish separation threshold in PCA space (soft penalty below)
}

def propose_next_point_week12_pca(
    ensemble, X_existing, Z_existing,
    best_x,
    rr,
    xi=0.004,
    dup_tol=1e-6,
    decoding=DECODING, mix=CAND_MIX, refine=REFINE,
    weights=SCORE_W, pen=PEN, top_report=10
):
    n_total = int(decoding["max_tokens"])
    n_best = int(mix["pca_local_best_frac"] * n_total)
    n_top  = int(mix["pca_local_top_frac"] * n_total)
    n_glo  = n_total - n_best - n_top

    # PCA anchor points
    z_best = to_pca(best_x.astype(np.float64))
    # weighted centroid of the top-ish region (via PCA weights from weighted_pca)
    z_top_centroid = np.sum(Z_existing * pca_w[:, None], axis=0)

    # Generate in PCA space (reduced dimension), then map back
    pca_sigma = float(mix["pca_trust"])
    z_best_cands = z_best + rr.normal(0.0, pca_sigma, size=(n_best, PCA_D))
    z_top_cands  = z_top_centroid + rr.normal(0.0, pca_sigma * 0.85, size=(n_top, PCA_D))

    # Global (raw uniform)
    global_raw = rr.random((n_glo, 4), dtype=np.float32)

    # Map PCA candidates back to raw
    raw_best = np.array([from_pca(z) for z in z_best_cands], dtype=np.float32)
    raw_top  = np.array([from_pca(z) for z in z_top_cands], dtype=np.float32)

    # Add small orthogonal noise to preserve exploration without high randomness
    # orth basis = remaining PCs
    V_orth = pca_V[:, PCA_D:]  # (4, 4-d)
    orth_sigma = float(mix["orth_noise"])
    if V_orth.shape[1] > 0:
        nb = raw_best.shape[0]
        nt = raw_top.shape[0]
        noise_b = rr.normal(0.0, orth_sigma, size=(nb, V_orth.shape[1])) @ V_orth.T
        noise_t = rr.normal(0.0, orth_sigma, size=(nt, V_orth.shape[1])) @ V_orth.T
        raw_best = clamp01(raw_best + noise_b.astype(np.float32))
        raw_top  = clamp01(raw_top  + noise_t.astype(np.float32))

    coarse = np.vstack([raw_best, raw_top, global_raw]).astype(np.float32)

    # --- Surrogate scoring ---
    X_cand_tensor = torch.tensor(coarse, dtype=torch.float32, device=device)
    mean, std = ensemble_predict(ensemble, X_cand_tensor)

    best_y = float(np.max(y_train))
    ei = expected_improvement(mean, std, best_y, xi=xi)
    pi = probability_of_improvement(mean, std, best_y, xi=0.0)

    mean_np = mean.detach().cpu().numpy()
    std_np  = std.detach().cpu().numpy()
    ei_np   = ei.detach().cpu().numpy()
    pi_np   = pi.detach().cpu().numpy()

    mean_n = norm01(mean_np)
    ei_n   = norm01(ei_np)
    pi_n   = norm01(pi_np)

    # Raw-space nearest penalty (extrapolation guard)
    d_raw = nearest_dist(coarse.astype(np.float64), X_existing.astype(np.float64))
    d_raw_n = norm01(d_raw)

    # PCA redundancy penalty: distance to nearest existing in PCA space
    Z_cand = np.array([to_pca(x.astype(np.float64)) for x in coarse], dtype=np.float64)
    dZ = nearest_dist(Z_cand, Z_existing)
    # soft redundancy penalty: strong penalty if closer than min_pca_sep
    min_sep = float(pen["min_pca_sep"])
    redund = np.maximum(0.0, (min_sep - dZ))
    redund_n = norm01(redund)

    score = (
        weights["w_mean"] * mean_n +
        weights["w_ei"]   * ei_n +
        weights["w_pi"]   * pi_n -
        float(pen["lambda_raw_near"]) * d_raw_n -
        float(pen["lambda_pca_redund"]) * redund_n
    )

    # Top-k seeds for refinement
    topk = int(refine["topk_seed"])
    seed_idx = np.argsort(score)[::-1][:topk]
    seeds = coarse[seed_idx]

    # --- refinement: jitter mainly in PCA space again ---
    per_seed = int(refine["per_seed"])
    pca_ref = float(refine["pca_refine"])
    orth_ref = float(refine["orth_refine"])

    refine_points = []
    for s in seeds:
        zs = to_pca(s.astype(np.float64))
        zj = zs + rr.normal(0.0, pca_ref, size=(per_seed, PCA_D))
        raw_j = np.array([from_pca(z) for z in zj], dtype=np.float32)

        # small orth refine noise
        if V_orth.shape[1] > 0:
            noise = rr.normal(0.0, orth_ref, size=(per_seed, V_orth.shape[1])) @ V_orth.T
            raw_j = clamp01(raw_j + noise.astype(np.float32))

        refine_points.append(raw_j)

    refine_all = np.vstack([coarse] + refine_points).astype(np.float32)

    X_ref_tensor = torch.tensor(refine_all, dtype=torch.float32, device=device)
    mean2, std2 = ensemble_predict(ensemble, X_ref_tensor)

    ei2 = expected_improvement(mean2, std2, best_y, xi=xi)
    pi2 = probability_of_improvement(mean2, std2, best_y, xi=0.0)

    mean2_np = mean2.detach().cpu().numpy()
    std2_np  = std2.detach().cpu().numpy()
    ei2_np   = ei2.detach().cpu().numpy()
    pi2_np   = pi2.detach().cpu().numpy()

    mean2_n = norm01(mean2_np)
    ei2_n   = norm01(ei2_np)
    pi2_n   = norm01(pi2_np)

    d_raw2 = nearest_dist(refine_all.astype(np.float64), X_existing.astype(np.float64))
    d_raw2_n = norm01(d_raw2)

    Z_ref = np.array([to_pca(x.astype(np.float64)) for x in refine_all], dtype=np.float64)
    dZ2 = nearest_dist(Z_ref, Z_existing)
    redund2 = np.maximum(0.0, (min_sep - dZ2))
    redund2_n = norm01(redund2)

    score2 = (
        weights["w_mean"] * mean2_n +
        weights["w_ei"]   * ei2_n +
        weights["w_pi"]   * pi2_n -
        float(pen["lambda_raw_near"]) * d_raw2_n -
        float(pen["lambda_pca_redund"]) * redund2_n
    )

    # Filter non-duplicates; greedy best-by-score
    keep = []
    for i in range(len(refine_all)):
        x = refine_all[i]
        if not is_duplicate(x, X_existing, tol=dup_tol):
            keep.append(i)
    if len(keep) == 0:
        chosen_i = int(np.argmax(score2))
    else:
        keep = np.array(keep, dtype=int)
        chosen_i = int(keep[np.argmax(score2[keep])])

    def pack(j):
        x = refine_all[j].astype(np.float64)
        return {
            "idx": int(j),
            "x_raw01": x,
            "mean": float(mean2_np[j]),
            "std": float(std2_np[j]),
            "ei": float(ei2_np[j]),
            "pi": float(pi2_np[j]),
            "score": float(score2[j]),
            "d_raw": float(d_raw2[j]),
            "dZ": float(dZ2[j]),
            "redund": float(redund2[j]),
        }

    chosen = pack(chosen_i)

    pool = np.arange(len(refine_all)) if len(keep) == 0 else keep
    pool_sorted_score = pool[np.argsort(score2[pool])[::-1]]
    pool_sorted_ei = pool[np.argsort(ei2_np[pool])[::-1]]

    report = {
        "top_by_ei": [pack(j) for j in pool_sorted_ei[:top_report]],
        "top_by_score": [pack(j) for j in pool_sorted_score[:top_report]],
    }

    meta = {
        "best_y": best_y,
        "n_coarse": int(n_total),
        "n_total_eval": int(len(refine_all)),
        "mix": mix,
        "refine": refine,
        "weights": weights,
        "pen": pen,
        "pca_d": PCA_D,
        "pca_explained": pca_expl,
        "selection": "GREEDY_BEST_BY_SCORE (PCA-SUBSPACE) + REDUNDANCY_REDUCTION",
        "domain_bounds": "[0,1]^4 (enforced)"
    }
    return chosen, report, meta

chosen, report, meta = propose_next_point_week12_pca(
    ensemble=ensemble,
    X_existing=X_train,
    Z_existing=Z_existing,
    best_x=current_best_x.astype(np.float32),
    rr=rng,
    xi=0.004,
    dup_tol=1e-6,
    top_report=10
)

next_x = clamp01(chosen["x_raw01"])
next_mean = chosen["mean"]
next_std  = chosen["std"]
next_ei   = chosen["ei"]
next_pi   = chosen["pi"]
next_score = chosen["score"]

# ----------------------------
# 9) Interpretability add-ons
# ----------------------------
nn_idx, nn_dist, nn_y = topk_nearest(next_x, X_train, y_train, k=3)

# ----------------------------
# 10) Report
# ----------------------------
print("\n================ WEEK 12 FUNCTION 4 RESULTS (v7 — PCA LENS) ================")

print("\nCURRENT BEST OBSERVED")
print("x_best =", fmt_x6(current_best_x), ", y_best =", f"{current_best_y:.6f}")

print("\nWEEK 12 SETTINGS (PCA lens)")
print("Domain bounds:", meta["domain_bounds"])
print("PCA_D:", meta["pca_d"])
print("Explained variance:", [float(v) for v in meta["pca_explained"]])
print("Candidate mix:", meta["mix"])
print("Refinement:", meta["refine"])
print("Score weights:", meta["weights"])
print("Penalties:", meta["pen"])
print("Selection:", meta["selection"])
print("Total coarse candidates:", meta["n_coarse"])
print("Total evaluated after refinement:", meta["n_total_eval"])

print("\nTOP-10 NON-DUPLICATE CANDIDATES (ranked by SCORE used for selection)")
for i, r in enumerate(report["top_by_score"], 1):
    print(
        f"{i:02d}) x={fmt_x6(r['x_raw01'])} | mean={r['mean']:.6f} std={r['std']:.6f} "
        f"EI={r['ei']:.6f} PI={r['pi']:.6f} SCORE={r['score']:.6f} "
        f"d_raw={r['d_raw']:.6f} dZ={r['dZ']:.6f} redund={r['redund']:.6f}"
    )

print("\nRECOMMENDED NEXT POINT (domain-safe; x_next in 6 decimals)")
print("x_next     =", fmt_x6(next_x))
print("mu(x_next) =", f"{next_mean:.6f}")
print("sigma      =", f"{next_std:.6f}")
print("EI         =", f"{next_ei:.6f}")
print("PI         =", f"{next_pi:.6f}")
print("SCORE      =", f"{next_score:.6f}")

print("\nINTERPRETABILITY CHECKS")
print("Nearest observed points to x_next (context):")
for rank, (ii, dd, yy) in enumerate(zip(nn_idx, nn_dist, nn_y), 1):
    print(f"  {rank}) idx={int(ii)}  x={fmt_x6(X_train[int(ii)])}  y={float(yy):.6f}  dist={float(dd):.6f}")

# ----------------------------
# 11) Week 12 reasoning prompts (explicit)
# ----------------------------
print("\nWEEK 12 REASONING (PCA Lens)")
print("- Strategy evolution: shifted from broad random exploration -> structured exploitation along learned principal directions.")
print("- Principal components: PCA (weighted toward higher y) identifies which combined variable movements explain most variance in good results.")
print("- Reduce/simplify: most candidate sampling occurs in the top PCA subspace (PC1..PC2), cutting redundant degrees of freedom.")
print("- Preserve exploration: small orthogonal noise + small global spill prevents missing a nearby basin.")
print("- Next/final round impact: this concentrates search on the dominant manifold while still allowing a controlled escape if the surrogate signals it.")
print("- PCA insight applied: we penalize redundancy in PCA space so new samples add information instead of re-sampling the same region.")


Current best index: 37
Current best X (clamped to [0,1]): [0.480706 0.444032 0.354963 0.354729]
Current best y: -0.81371612670717

=== WEEK 12 PCA SUMMARY ===
Explained variance ratio: [0.5034494847162322, 0.30668751182691567, 0.1097204973401022, 0.08014250596140447]
Using PCA_D = 2 principal components.
Weighted PCA mean: [0.492775, 0.435379, 0.361788, 0.362693]
PC1 vector: [-0.335586, 0.144035, 0.098109, -0.925749]
PC2 vector: [0.665322, 0.060770, 0.727843, -0.154590]

================ WEEK 12 FUNCTION 4 RESULTS (v7 — PCA LENS) ================

CURRENT BEST OBSERVED
x_best = [0.480706, 0.444032, 0.354963, 0.354729] , y_best = -0.813716

WEEK 12 SETTINGS (PCA lens)
Domain bounds: [0,1]^4 (enforced)
PCA_D: 2
Explained variance: [0.5034494847162322, 0.30668751182691567, 0.1097204973401022, 0.08014250596140447]
Candidate mix: {'pca_local_best_frac': 0.72, 'pca_local_top_frac': 0.18, 'global_frac': 0.1, 'pca_trust': 0.075, 'orth_noise': 0.012}
Refinement: {'topk_seed': 180, 'per_seed': 1